# Silver Layer - Data Cleaning & Transformation

In [0]:
# ============================================================
# Notebook : 02_silver_layer
# Project  : Supply Chain Control Tower
# Author   : Nisha Sorallikar
# Purpose  : Clean, transform, and validate Bronze data
#            before preparing it for analytics.
# ============================================================

In [0]:
# ---------------------------------------
# Step 1 : Read Bronze Delta Table
# ---------------------------------------

# Read the Bronze Delta table.
# This table is the source for all transformations in the Silver layer.

df_silver = spark.table("dev_project.default.bronze_supply_chain")

# Display the first 5 records to verify the data.

df_silver.show(5)

# ---------------------------------------
# Step 2 : Verify Bronze Schema
# ---------------------------------------

# Display the schema of the Bronze DataFrame.
# This helps us identify the columns and their data types
# before applying any transformations.

df_silver.printSchema()

# ---------------------------------------
# Step 3 : Convert Date Columns
# ---------------------------------------

from pyspark.sql.functions import to_timestamp, col

# Convert order_date_dateorders from string to timestamp.
df_silver = df_silver.withColumn(
    "order_date",
    to_timestamp(col("order_date_dateorders"), "M/d/yyyy H:mm")
)

# Convert shipping_date_dateorders from string to timestamp.
df_silver = df_silver.withColumn(
    "shipping_date",
    to_timestamp(col("shipping_date_dateorders"), "M/d/yyyy H:mm")
)

# ---------------------------------------
# Step 4 : Verify Date Conversion
# ---------------------------------------

df_silver.select(
    "order_date_dateorders",
    "order_date",
    "shipping_date_dateorders",
    "shipping_date"
).show(5, truncate=False)

# ---------------------------------------
# Step 5 : Remove Unnecessary Columns
# ---------------------------------------

# Remove columns that are no longer required.
# - product_description contains only NULL values.
# - order_date_dateorders and shipping_date_dateorders
#   are replaced by timestamp columns.

df_silver = df_silver.drop(
    "product_description",
    "order_date_dateorders",
    "shipping_date_dateorders"
)

# Verify the updated schema.

df_silver.printSchema()

# ---------------------------------------
# Step 6 : Validate NULL Values
# ---------------------------------------

from pyspark.sql.functions import col, count, when

null_df = df_silver.select([
    count(when(col(column).isNull(), column)).alias(column)
    for column in df_silver.columns
])

null_df.show(truncate=False)

# ---------------------------------------
# Step 7 : Handle NULL Values
# ---------------------------------------

from pyspark.sql.functions import col, when

# Replace NULL customer last names with "Unknown"

df_silver = df_silver.fillna({
    "customer_lname": "Unknown"
})

# Replace NULL customer zip codes with 0

df_silver = df_silver.fillna({
    "customer_zipcode": 0
})

# Drop the order_zipcode column because
# approximately 86% of its values are NULL
# and it is not required for reporting.

df_silver = df_silver.drop("order_zipcode")

# ---------------------------------------
# Step 8 : Verify NULL Handling
# ---------------------------------------

from pyspark.sql.functions import col, count, when

df_silver.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df_silver.columns
]).show(truncate=False)

# ---------------------------------------
# Step 9 : Validate Silver Data
# ---------------------------------------

# Total number of records
print(f"Total Rows : {df_silver.count()}")

# Check duplicate records
print(f"Distinct Rows : {df_silver.distinct().count()}")

# Display schema
df_silver.printSchema()

# Preview cleaned data
df_silver.show(5)

# ---------------------------------------
# Step 10 : Create Silver Delta Table
# ---------------------------------------

# Write the cleaned DataFrame to the Silver layer
# in Delta format.

(
    df_silver.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable("dev_project.default.silver_supply_chain")
)

# ---------------------------------------
# Step 11 : Verify Silver Table
# ---------------------------------------

spark.sql("""
SELECT COUNT(*) AS total_records
FROM dev_project.default.silver_supply_chain
""").show()


# ---------------------------------------
# Data Quality Checks
# ---------------------------------------

# ---------------------------------------
# Check 1 : Row Count Validation
# ---------------------------------------

bronze_count = spark.table("dev_project.default.bronze_supply_chain").count()
silver_count = spark.table("dev_project.default.silver_supply_chain").count()

print(f"Bronze Row Count : {bronze_count}")
print(f"Silver Row Count : {silver_count}")

if bronze_count == silver_count:
    print("✅ PASS - Row counts match")
else:
    print("❌ FAIL - Row counts do not match")

# ---------------------------------------
# Check 2 : Duplicate Order Item ID
# ---------------------------------------

duplicate_count = (
    spark.table("dev_project.default.silver_supply_chain")
         .groupBy("order_item_id")
         .count()
         .filter("count > 1")
         .count()
)

print(f"Duplicate Order Item IDs : {duplicate_count}")

if duplicate_count == 0:
    print("✅ PASS - No duplicate order_item_id values")
else:
    print("❌ FAIL - Duplicate order_item_id values found")

# ---------------------------------------
# Check 3 : Date Validation
# ---------------------------------------

invalid_dates = (
    spark.table("dev_project.default.silver_supply_chain")
         .filter("shipping_date < order_date")
         .count()
)

print(f"Invalid Date Records : {invalid_dates}")

if invalid_dates == 0:
    print("✅ PASS - All shipping dates are valid")
else:
    print("❌ FAIL - Invalid shipping dates found")

# ---------------------------------------
# Check 4 : NULL Validation
# ---------------------------------------

from pyspark.sql.functions import col, count, when

spark.table("dev_project.default.silver_supply_chain").select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in spark.table("dev_project.default.silver_supply_chain").columns
]).show(truncate=False)


# ---------------------------------------
# Check 5 : Quantity Validation
# ---------------------------------------

invalid_quantity = (
    spark.table("dev_project.default.silver_supply_chain")
         .filter("order_item_quantity <= 0")
         .count()
)

print(f"Invalid Quantity Records : {invalid_quantity}")

if invalid_quantity == 0:
    print("✅ PASS - All quantities are valid")
else:
    print("❌ FAIL - Invalid quantities found")